# Create SQLite Database from CSV Files

This notebook creates a SQLite database named `baseball.db` with three tables based on the provided CSV files: `people.csv`, `teams.csv`, and `Batting(1).csv`. The tables will have correct data types and relational constraints.

In [1]:
import pandas as pd
import sqlite3
from sqlalchemy import create_engine, text

## Load CSV Files into Pandas DataFrames

Read the three CSV files into separate pandas DataFrames with appropriate data types.

In [3]:
# Load the DataFrames
people_df = pd.read_csv('people.csv')
teams_df = pd.read_csv('teams.csv')
batting_df = pd.read_csv('Batting(1).csv')

print("DataFrames loaded successfully!")
print(f"People: {people_df.shape}")
print(f"Teams: {teams_df.shape}")
print(f"Batting: {batting_df.shape}")

DataFrames loaded successfully!
People: (24270, 25)
Teams: (3614, 48)
Batting: (128598, 22)


## Inspect and Adjust Data Types

Check the data types of the loaded DataFrames.

In [7]:
print("People columns:", list(people_df.columns))
print("Teams columns:", list(teams_df.columns))
print("Batting columns:", list(batting_df.columns))

People columns: ['ID', 'playerID', 'birthYear', 'birthMonth', 'birthDay', 'birthCity', 'birthCountry', 'birthState', 'deathYear', 'deathMonth', 'deathDay', 'deathCountry', 'deathState', 'deathCity', 'nameFirst', 'nameLast', 'nameGiven', 'weight', 'height', 'bats', 'throws', 'debut', 'bbrefID', 'finalGame', 'retroID']
Teams columns: ['yearID', 'lgID', 'teamID', 'franchID', 'divID', 'Rank', 'G', 'Ghome', 'W', 'L', 'DivWin', 'WCWin', 'LgWin', 'WSWin', 'R', 'AB', 'H', '2B', '3B', 'HR', 'BB', 'SO', 'SB', 'CS', 'HBP', 'SF', 'RA', 'ER', 'ERA', 'CG', 'SHO', 'SV', 'IPouts', 'HA', 'HRA', 'BBA', 'SOA', 'E', 'DP', 'FP', 'name', 'park', 'attendance', 'BPF', 'PPF', 'teamIDBR', 'teamIDlahman45', 'teamIDretro']
Batting columns: ['playerID', 'yearID', 'stint', 'teamID', 'lgID', 'G', 'AB', 'R', 'H', '2B', '3B', 'HR', 'RBI', 'SB', 'CS', 'BB', 'SO', 'IBB', 'HBP', 'SH', 'SF', 'GIDP']


## Create SQLite Database and Define Table Schemas

Create the SQLite database and define the table schemas with primary and foreign key constraints.

In [8]:
teams_sql_dtypes = {
    'yearID': 'INTEGER',
    'lgID': 'TEXT',
    'teamID': 'TEXT',
    'franchID': 'TEXT',
    'divID': 'TEXT',
    'Rank': 'INTEGER',
    'G': 'INTEGER',
    'Ghome': 'INTEGER',
    'W': 'INTEGER',
    'L': 'INTEGER',
    'DivWin': 'TEXT',
    'WCWin': 'TEXT',
    'LgWin': 'TEXT',
    'WSWin': 'TEXT',
    'R': 'INTEGER',
    'AB': 'INTEGER',
    'H': 'INTEGER',
    '2B': 'INTEGER',
    '3B': 'INTEGER',
    'HR': 'INTEGER',
    'BB': 'INTEGER',
    'SO': 'INTEGER',
    'SB': 'INTEGER',
    'CS': 'INTEGER',
    'HBP': 'INTEGER',
    'SF': 'INTEGER',
    'RA': 'INTEGER',
    'ER': 'INTEGER',
    'ERA': 'REAL',
    'CG': 'INTEGER',
    'SHO': 'INTEGER',
    'SV': 'INTEGER',
    'IPouts': 'INTEGER',
    'HA': 'INTEGER',
    'HRA': 'INTEGER',
    'BBA': 'INTEGER',
    'SOA': 'INTEGER',
    'E': 'INTEGER',
    'DP': 'INTEGER',
    'FP': 'REAL',
    'name': 'TEXT',
    'park': 'TEXT',
    'attendance': 'INTEGER',
    'BPF': 'INTEGER',
    'PPF': 'INTEGER',
    'teamIDBR': 'TEXT',
    'teamIDlahman45': 'TEXT',
    'teamIDretro': 'TEXT'
}

batting_sql_dtypes = {
    'playerID': 'TEXT',
    'yearID': 'INTEGER',
    'stint': 'INTEGER',
    'teamID': 'TEXT',
    'lgID': 'TEXT',
    'G': 'INTEGER',
    'AB': 'INTEGER',
    'R': 'INTEGER',
    'H': 'INTEGER',
    '2B': 'INTEGER',
    '3B': 'INTEGER',
    'HR': 'INTEGER',
    'RBI': 'INTEGER',
    'SB': 'INTEGER',
    'CS': 'INTEGER',
    'BB': 'INTEGER',
    'SO': 'INTEGER',
    'IBB': 'INTEGER',
    'HBP': 'INTEGER',
    'SH': 'INTEGER',
    'SF': 'INTEGER',
    'GIDP': 'INTEGER'
}

## Insert DataFrames into SQLite Tables

Insert the data from each DataFrame into its corresponding SQLite table.

In [9]:
# Insert data into tables
people_df.to_sql('people', engine, if_exists='append', index=False)
teams_df.to_sql('teams', engine, if_exists='append', index=False)
batting_df.to_sql('batting', engine, if_exists='append', index=False)

print("Data inserted successfully!")

Data inserted successfully!


## Verify Data and Constraints in Database

Query the database to confirm that the data has been loaded correctly and that the constraints are in place.

In [10]:
# Verify data insertion
with engine.connect() as conn:
    # Check row counts
    people_count = conn.execute(text('SELECT COUNT(*) FROM people')).fetchone()[0]
    teams_count = conn.execute(text('SELECT COUNT(*) FROM teams')).fetchone()[0]
    batting_count = conn.execute(text('SELECT COUNT(*) FROM batting')).fetchone()[0]
    
    print(f"People table: {people_count} rows")
    print(f"Teams table: {teams_count} rows")
    print(f"Batting table: {batting_count} rows")
    
    # Check a sample query with joins
    sample_query = conn.execute(text('''
        SELECT p.nameFirst, p.nameLast, b.yearID, b.HR, t.name
        FROM batting b
        JOIN people p ON b.playerID = p.playerID
        JOIN teams t ON b.yearID = t.yearID AND b.teamID = t.teamID
        LIMIT 5
    ''')).fetchall()
    
    print("\nSample joined data:")
    for row in sample_query:
        print(row)

print("Database verification complete!")

People table: 24270 rows
Teams table: 3614 rows
Batting table: 128598 rows

Sample joined data:
('David', 'Aardsma', 2004, 0, 'San Francisco Giants')
('David', 'Aardsma', 2006, 0, 'Chicago Cubs')
('David', 'Aardsma', 2007, 0, 'Chicago White Sox')
('David', 'Aardsma', 2008, 0, 'Boston Red Sox')
('David', 'Aardsma', 2009, 0, 'Seattle Mariners')
Database verification complete!
